## Using Agents in LlamaIndex

### Initialising Agents

In [1]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.agent.workflow import AgentWorkflow, FunctionAgent, ReActAgent
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import FunctionTool
from llama_index.core.workflow import Context
from llama_index.core.tools import QueryEngineTool
from llama_index.core import VectorStoreIndex, Document
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [2]:
# Define Sample Tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers and returns the resulting integer"""
    return a * b

In [ ]:
# Initialize llm
llm = HuggingFaceInferenceAPI(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    token="")

# Initialize agent
agent = AgentWorkflow.from_tools_or_functions(
    [FunctionTool.from_defaults(multiply)],
    llm=llm
)

In [4]:
# Stateless
response = await agent.run("What is 2 times 2?")
print(response)

4


In [5]:
# Remembering state
ctx = Context(agent)
response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
print(response)

Your name is Bob.


### Creating RAG Agents with QueryEngineTools

In [6]:
# Create a simple document
document = Document(
    text="Bob is a friendly software engineer who enjoys helping people."
)

# Create an index from the document
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents([document], embed_model=embed_model)

In [7]:
# Create a query engine
query_engine = index.as_query_engine(
    llm=llm,
    embed_model=embed_model,
    similarity_top_k=3
)

# Convert the query engine into a tool
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="persona_search",
    description="Search the database for persona descriptions.",
    return_direct=False,
)

# Create an agent that can use the query tool
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    [query_engine_tool],
    llm=llm,
    system_prompt=(
        "You are a helpful assistant that has access to a database "
        "containing persona descriptions."
    )
)

In [8]:
response = await query_engine_agent.run(
    "What do you know about Bob?"
)

print(response)

Bob is likely to be enthusiastic about assisting individuals with their software-related queries.


### Creating Multi-agent systems

In [13]:
# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b

In [14]:
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm
)

In [15]:
# Create and run the workflow
agent = AgentWorkflow(
    agents=[calculator_agent, query_agent], root_agent="calculator"
)

# Run the system
response = await agent.run(user_msg="Can you add 5 and 3?")
print(response)

The result of adding 5 and 3 is 8.
